# CueNote Phase 9E-H Colab Training

Use this notebook for dataset import, checkpoint/resume, layout training, symbol training, ONNX export, parity checks, and artifact packaging.

User steps before Run all:

1. Runtime -> Change runtime type -> GPU for real training.
2. Approve Google Drive mount.
3. Confirm dataset terms and license evidence.
4. Set `CUENOTE_REPO_URL` if the repository is not already present in Colab.
5. Choose `CUENOTE_RUN_MODE`.

Default `SMOKE` mode prepares and validates data only. It does not claim GPU training or candidate metrics.

In [ ]:
# Runtime / Drive setup. Run all starts here.
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
os.environ.setdefault('CUENOTE_DRIVE_ROOT', '/content/drive/MyDrive/CueNote')
os.environ.setdefault('CUENOTE_RUN_MODE', 'SMOKE')  # SMOKE | LAYOUT_TRAIN | SYMBOL_TRAIN | FULL_PIPELINE
print('Runtime mode:', os.environ['CUENOTE_RUN_MODE'])
print('Drive root:', os.environ['CUENOTE_DRIVE_ROOT'])

In [ ]:
# Clone or reuse repository.
import os, subprocess
from pathlib import Path

repo_url = os.environ.get('CUENOTE_REPO_URL', '').strip()
repo_dir = Path('/content/CueNote')
if repo_dir.exists() and (repo_dir / '.git').exists():
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only'], check=False)
elif repo_url:
    subprocess.run(['git', 'clone', repo_url, str(repo_dir)], check=True)
else:
    raise RuntimeError('Set CUENOTE_REPO_URL or upload/clone the repository to /content/CueNote before Run all.')
print('Repository:', repo_dir)
print(subprocess.check_output(['git', '-C', str(repo_dir), 'rev-parse', '--short', 'HEAD'], text=True))

In [ ]:
# Dependency install. Versions are pinned in requirements-colab.txt.
import subprocess, sys
from pathlib import Path
repo_dir = Path('/content/CueNote')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(repo_dir / 'ai-training/requirements-colab.txt')], check=True)

In [ ]:
# GPU/environment report. Full training requires CUDA GPU.
import torch, platform, shutil
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', torch.cuda.get_device_properties(0).total_memory)
else:
    print('No GPU allocated. Use Runtime -> Change runtime type -> GPU. SMOKE mode may continue; full training must not be marked PASS.')
print('Disk:', shutil.disk_usage('/content'))

In [ ]:
# Phase 9 pipeline entry. This creates Drive directories, run-state.json, checkpoint metadata, reports, ONNX, and artifact zip.
import os, subprocess, sys
from pathlib import Path
repo_dir = Path('/content/CueNote')
cmd = [
    sys.executable,
    str(repo_dir / 'ai-training/python/phase9_colab_entry.py'),
    '--run-mode', os.environ['CUENOTE_RUN_MODE'],
    '--drive-root', os.environ['CUENOTE_DRIVE_ROOT'],
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

## After Run all

Check Drive paths:

- `/content/drive/MyDrive/CueNote/run-state.json`
- `/content/drive/MyDrive/CueNote/checkpoints/layout` and `checkpoints/symbol`
- `/content/drive/MyDrive/CueNote/reports`
- `/content/drive/MyDrive/CueNote/artifacts`

If the session disconnects, rerun the notebook. Existing compatible checkpoint/resume metadata is reused. If config, dataset, or taxonomy checksums change, start a new run instead of silently resuming.

Download the artifact zip and run local validation before installing it into the Web PWA runtime.